# 3. External data

Reads files that never went through a questionnaire - published indicator
tables from another source, one folder per chapter under
`DATA COLLECTOR\external data\<Chapter>\` - and writes each chapter's
rows to `<Chapter>_EN_external.xlsx`, already in English.

```
DATA COLLECTOR\external data\<Chapter>\*.xlsx
        -> DATA COLLECTOR\external data\<Chapter>\*_reshaped.xlsx   (the questionnaire-layout copy, beside the source)
        -> merged_long_files\<Chapter>_EN_external.xlsx               (read by notebook 4)
```

**Runs before notebook 4, between it and notebook 2** - this notebook does
none of the translation itself. `<Chapter>_EN_external.xlsx` is an INPUT to
notebook 4 the same shape as `<Chapter>_EN_questionnaires.xlsx` is, except
notebook 4 also has to give it an Arabic form, since (unlike a real English
questionnaire) nothing else in this project has ever written this content
in Arabic. Splitting the work this way keeps every translation and
dictionary-gap decision in the one notebook that already owns them, rather
than this notebook keeping its own separate copy of that machinery.

## Reshaping into the questionnaire layout

Follows the `reshape-external-data-questionnaire-layout` skill:
`save_questionnaire_layout()` writes one reshaped workbook per source file,
beside the source file, styled to match a real questionnaire's layout -
title row, bold bordered header, frozen panes - built from the SAME rows
`read_external_sheet()` already produced for `<Chapter>_EN_external.xlsx`,
not a second reading of the sheet, so the deliverable can never disagree
with what actually gets written.

## Reading a source sheet - a fixed shape, not an inferred one

**As of 24 September 2026, the source file's own layout was changed to
carry two clearly marked blocks, and this notebook was rewritten to match
it exactly rather than infer anything.** Every sheet now stacks:

- **`index = 1`** - the data, already long: one row per country-year, with
  the indicator named either by its own value column's header (`index,
  Country, Year, <name>`) or by an explicit `Indicator` column (`index,
  Indicator, Country, Year, ...`) ahead of a Male/Female/Total split or a
  single, inconsistently-named value column.
- **`index = 2`** - that data's citation: `index, indicator, year, source`,
  no `Country` of its own, so one citation applies to every country
  reporting that indicator-year. Matched to the data block by
  `(Indicator, Year)`.

Both header rows are found by their own literal `"index"` label - the same
convention notebook 1 already reads a raw questionnaire's two header rows
with (`data_header_row`, `source_header_row`) - so nothing here scans for a
shape or guesses a position the way the old version had to. A sheet whose
`index = 1` header does not start with `Country` or `Indicator`, or whose
value columns are neither `Male`/`Female`/`Total` nor a single column, is
refused entirely and reported why - there is no longer a partial column to
leave out on its own, since there is no shape left to be partially sure of.

**Before 24 September 2026**, the source file mixed group-label rows,
merged header cells and a generic "value" column with no name of its own,
discovered by reading the real file rather than guessing at a spec - see
Known issues in `../CLAUDE.md` for what that cost when it went wrong (`4.10`
losing 781 otherwise-good figures to one stray, unheaded value). That
history no longer describes this notebook's code, only why a fixed,
two-block layout was worth asking for.

## Marking what came from outside

Every row written gets `Data Origin` = `External` - blank for every row
that was already there. Notebook 5's row and breakdown columns are a fixed,
named list (`ROW_COLUMNS`, `BREAKDOWN_COLUMNS`); `Data Origin` is in
neither, so tabulations already ignore it without anything there having to
change. The reshaped questionnaire-layout copy is a human-readable side
deliverable only - nothing in the real pipeline reads it back.

## Running it

Run the cells through **Run** below. `<Chapter>_EN_external.xlsx` is
overwritten in full each run - this notebook owns that file outright, so
there is nothing of a previous run to strip first the way notebook 4 has to
for the Arabic side. Report: sheets read, sheets it could not confidently
read (and why). No dictionary gaps are found or filled here - notebook 4
finds and fills whatever this notebook's output needs.


In [ ]:
"""
CELL: Imports and logging setup.
"""
from collections import OrderedDict, defaultdict
import logging
import re
from pathlib import Path

import openpyxl
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter
import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("compendium")


## Config

In [ ]:
"""
CELL: Configuration - paths, the chapters this looks for, and the new
column that marks a row as coming from outside the questionnaires.
"""
DATA_COLLECTOR_PATH = Path.home() / "OneDrive - United Nations" / "Desktop" / "DSS" / "DATA COLLECTOR"
EXTERNAL_DATA_PATH = DATA_COLLECTOR_PATH / "external data"
COMPENDIUM_PATH = Path.home() / "OneDrive - United Nations" / "Desktop" / "DSS" / "COMPENDIUM-ARAB SOCIETY"
LONG_FILES_PATH = COMPENDIUM_PATH / "merged_long_files"

# Leave as None to check every chapter with an external-data subfolder, or
# restrict e.g. ["Health"].
CHAPTERS = None

# Marks a row as not from the questionnaires. Blank for every existing row -
# nothing already in a long file is touched or backfilled with this - and set
# only on rows this notebook itself writes. Not in notebook 5's ROW_COLUMNS
# or BREAKDOWN_COLUMNS, so tabulations never see it; nothing there has to
# change for that to stay true.
ORIGIN_COLUMN_EN = "Data Origin"
ORIGIN_EXTERNAL_EN = "External"

# The six domains - both here and as `external data\<name>\` subfolder names.
KNOWN_CHAPTERS = ["Health", "Population", "Education", "Labor", "Poverty", "Housing"]


def external_file_path(chapter):
    """Where this chapter's extracted-but-not-yet-translated rows land -
    an INPUT to notebook 4, the same idea as <Chapter>_EN_questionnaires.xlsx
    except notebook 4 also has to give this one an Arabic form."""
    return LONG_FILES_PATH / f"{chapter}_EN_external.xlsx"


# Everything this run fixed, inferred, or added to the dictionary on its own
# goes to pipeline_changes_<Chapter>.txt; everything it left alone, and why,
# goes to need manual intervention_<Chapter>.txt. See save_inconsistencies()
# below.


def changes_log_path(chapter):
    """Where one chapter's own auto-applied changes live - what this run
    corrected, inferred, or added to the dictionary on its own."""
    return COMPENDIUM_PATH / f"pipeline_changes_{chapter}.txt"


def manual_intervention_log_path(chapter):
    """Where one chapter's own open findings live - what this run left
    alone, and why, for a person to look at."""
    return COMPENDIUM_PATH / f"need manual intervention_{chapter}.txt"


GENERAL_CHANGES_LOG_PATH = COMPENDIUM_PATH / "pipeline_changes_general.txt"
GENERAL_MANUAL_LOG_PATH = COMPENDIUM_PATH / "need manual intervention_general.txt"

# A record's own `kind` decides which of the two files it lands in - fixed,
# inferred, or added to the dictionary goes to pipeline_changes_<Chapter>.txt;
# everything else (a gap, a skip, a collision) goes to
# need manual intervention_<Chapter>.txt. Anything not listed here defaults
# to manual intervention, on purpose - a new kind nobody has classified yet
# should surface for a person to see, not disappear into "already handled".
CHANGE_KINDS = {
    "column name corrected",
    "value corrected",
    "value needed correcting",
    "reused a close Arabic spelling",
    "dictionary entry added",
}


def _write_section(path, title, section, section_text):
    """Replace one named section in one file, in place, leaving every other
    section exactly as it was. The mechanic every chapter file and the
    general one share - split what's there into sections by name, replace
    this one, rebuild in section order so the file reads the same whatever
    order the notebooks last ran in."""
    header = [title, "=" * 78, ""]

    sections = {}
    if path.exists():
        existing = path.read_text(encoding="utf-8")
        parts = re.split(r"^### (.+?) ###$", existing, flags=re.M)
        for name, text in zip(parts[1::2], parts[2::2]):
            sections[name] = f"### {name} ###{text.rstrip()}"
    sections[section] = section_text

    body_text = "\n\n".join(sections[name] for name in sorted(sections))
    path.write_text("\n".join(header).rstrip("\n") + "\n\n" + body_text + "\n",
                    encoding="utf-8")


def save_inconsistencies(section, records, chapters=None):
    """Write this notebook's findings for this section, split into two files
    per chapter - pipeline_changes_<Chapter>.txt for everything this run
    fixed, inferred, or added to the dictionary on its own, and
    need manual intervention_<Chapter>.txt for everything it left alone and
    why. A finding with no chapter (a dictionary-level problem, not a
    source-data one) goes to the matching _general file instead.

    `chapters` is every chapter this call actually covers, independent of
    whether any of them have a finding - pass it explicitly so a clean
    chapter still gets both files correctly replaced with "Nothing found"
    rather than left showing whatever an earlier, unrelated run left there.
    """
    stamp = pd.Timestamp.now().strftime("%d %B %Y, %H:%M")
    marker = f"### {section} ###"

    WHERE = ["country", "indicator", "year", "sex", "age_group",
             "nationality", "area", "file", "sheet", "row"]

    def render(chapter_records):
        body = [marker, f"    last run {stamp}", ""]
        if not chapter_records:
            body += ["    Nothing found.", ""]
        else:
            frame = pd.DataFrame(chapter_records)
            for kind, group in frame.groupby("kind", sort=False):
                body.append(f"  {kind.upper()}  ({len(group)})")
                for _, row in group.iterrows():
                    def show(value):
                        if isinstance(value, float) and float(value).is_integer():
                            return str(int(value))
                        return str(value)
                    where = " · ".join(
                        show(row[f]) for f in WHERE
                        if f in row and pd.notna(row[f]) and str(row[f]) != "")
                    body.append(f"      {where}" if where else "      -")
                    body.append(f"          {row['detail']}")
                body.append("")
        return "\n".join(body)

    def split(chapter_records):
        changes = [r for r in chapter_records if r["kind"] in CHANGE_KINDS]
        manual = [r for r in chapter_records if r["kind"] not in CHANGE_KINDS]
        return changes, manual

    by_chapter = defaultdict(list)
    general = []
    for record in records:
        chapter = record.get("chapter")
        if chapter in (None, "", "-"):
            general.append(record)
        else:
            by_chapter[str(chapter)].append(record)

    covered = {str(c) for c in (chapters or [])} | set(by_chapter)

    paths = []
    for chapter in sorted(covered):
        changes, manual = split(by_chapter.get(chapter, []))
        _write_section(changes_log_path(chapter), f"PIPELINE CHANGES — {chapter}",
                       section, render(changes))
        _write_section(manual_intervention_log_path(chapter),
                       f"NEED MANUAL INTERVENTION — {chapter}", section, render(manual))
        paths += [changes_log_path(chapter), manual_intervention_log_path(chapter)]

    if general or not covered:
        changes, manual = split(general)
        _write_section(GENERAL_CHANGES_LOG_PATH, "PIPELINE CHANGES — general",
                       section, render(changes))
        _write_section(GENERAL_MANUAL_LOG_PATH, "NEED MANUAL INTERVENTION — general",
                       section, render(manual))
        paths += [GENERAL_CHANGES_LOG_PATH, GENERAL_MANUAL_LOG_PATH]

    return paths, len(records)


def external_chapters():
    """Chapters with a folder under external data\\ holding at least one .xlsx."""
    if not EXTERNAL_DATA_PATH.exists():
        logger.warning(f"{EXTERNAL_DATA_PATH} does not exist - nothing to read")
        return []
    found = []
    names = CHAPTERS if CHAPTERS else KNOWN_CHAPTERS
    for name in names:
        folder = EXTERNAL_DATA_PATH / name
        if folder.exists() and any(f for f in folder.glob("*.xlsx") if not f.name.startswith("~$")):
            found.append(name)
    return found


## Reading the two blocks a sheet is now guaranteed to have


In [ ]:
"""
CELL: The building blocks read_external_sheet() (below) is made of - finding
the two "index" header rows, splitting the sheet on them, and reading each
block in whichever of its few known shapes it is. Nothing here scans for a
shape or guesses a position; every sheet either matches one of these shapes
exactly or is refused and reported why.
"""

SEX_LABELS = {"male": "Male", "female": "Female", "total": "Both sexes"}


def is_blank(value):
    # pd.read_excel gives a blank cell as float NaN, not None - is_blank()
    # used to miss that case, so a genuinely empty Source cell came back
    # as the literal text "nan" (str(float("nan"))) instead of staying blank.
    return pd.isna(value) or (isinstance(value, str) and value.strip() == "")


def clean_text(value):
    return "" if is_blank(value) else str(value).strip()


def collapse_whitespace(text):
    """A citation or indicator name wrapped across lines in its source
    cell, read back as one cell with an embedded newline - flattened to one
    line so it never silently breaks a downstream regex (`.` does not cross
    a newline without re.DOTALL) or reads oddly in a chart title."""
    return re.sub(r"\s+", " ", str(text)).strip()


def find_index_header_rows(raw_sheet):
    """Both rows whose own first cell reads exactly "index" - the data
    block's header, then the source block's header. A sheet with any other
    count is not this shape at all and is refused by the caller."""
    return raw_sheet.index[
        raw_sheet[0].apply(lambda v: clean_text(v).lower()) == "index"
    ].tolist()


def split_blocks(raw_sheet, header_rows):
    """Each header row's own values, and the body rows between it and the
    next header (or the end of the sheet) - the block's index numbers are
    read from that body by the caller, not assumed from position."""
    blocks = []
    for i, header_row in enumerate(header_rows):
        header = [clean_text(h) for h in raw_sheet.iloc[header_row].tolist()]
        end = header_rows[i + 1] if i + 1 < len(header_rows) else len(raw_sheet)
        body = raw_sheet.iloc[header_row + 1: end]
        blocks.append((header, body))
    return blocks


def read_data_block(header, body):
    """The index=1 block, in whichever of the two known shapes it is:

      - index, Country, Year, <the indicator's own name as this column's
        header>
      - index, Indicator, Country, Year, Male, Female, Total
      - index, Indicator, Country, Year, <a single, inconsistently named
        value column - "value", "Value", ...>

    Returns (rows, note). An unrecognized shape returns no rows and a note
    explaining why - refusing the whole sheet, since there is no longer a
    partial column to leave out on its own.
    """
    body = body[pd.to_numeric(body[0], errors="coerce") == 1]
    if body.empty:
        return [], "no index=1 rows found"

    if header[1].lower() == "country":
        indicator = collapse_whitespace(header[3])
        rows = []
        for _, row in body.iterrows():
            year = pd.to_numeric(row[2], errors="coerce")
            value = pd.to_numeric(row[3], errors="coerce")
            if pd.isna(year) or pd.isna(value):
                continue
            rows.append({"Country": clean_text(row[1]), "Year": int(year),
                        "Value": float(value), "Indicator": indicator, "Sex": None})
        return rows, f"{len(rows):,} row(s), indicator named by its own column"

    if header[1].lower() == "indicator":
        rest = header[4:]
        rest_lower = [c.lower() for c in rest]
        if sorted(rest_lower) == ["female", "male", "total"]:
            positions = {c.lower(): 4 + i for i, c in enumerate(rest)}
            rows = []
            for _, row in body.iterrows():
                year = pd.to_numeric(row[3], errors="coerce")
                if pd.isna(year):
                    continue
                indicator = collapse_whitespace(row[1])
                for label, position in positions.items():
                    value = pd.to_numeric(row[position], errors="coerce")
                    if pd.isna(value):
                        continue
                    rows.append({"Country": clean_text(row[2]), "Year": int(year),
                                "Value": float(value), "Indicator": indicator,
                                "Sex": SEX_LABELS[label]})
            return rows, f"{len(rows):,} row(s), Male/Female/Total"
        if len(rest) == 1:
            rows = []
            for _, row in body.iterrows():
                year = pd.to_numeric(row[3], errors="coerce")
                value = pd.to_numeric(row[4], errors="coerce")
                if pd.isna(year) or pd.isna(value):
                    continue
                rows.append({"Country": clean_text(row[2]), "Year": int(year),
                            "Value": float(value),
                            "Indicator": collapse_whitespace(row[1]), "Sex": None})
            return rows, f"{len(rows):,} row(s), single value column {rest[0]!r}"
        return [], (f"index=1 header has an unrecognized shape after "
                    f"Indicator/Country/Year: {rest!r} - skipped")

    return [], f"index=1 header does not start with Country or Indicator: {header!r} - skipped"


def read_source_block(header, body):
    """The index=2 block - always index, indicator, year, source. No
    Country of its own, so one citation applies to every country reporting
    that indicator-year - matched to the data block on (Indicator, Year)."""
    body = body[pd.to_numeric(body[0], errors="coerce") == 2]
    if body.empty:
        return pd.DataFrame(columns=["Indicator", "Year", "Source"])
    return pd.DataFrame({
        "Indicator": [collapse_whitespace(v) for v in body[1]],
        "Year": pd.to_numeric(body[2], errors="coerce"),
        "Source": [None if is_blank(v) else clean_text(v) for v in body[3]],
    }).dropna(subset=["Year"])


## read_external_sheet() - one sheet's two blocks, joined into rows


In [ ]:
"""
CELL: read_external_sheet() - turn one raw sheet's two blocks into long
rows with their source already attached, or refuse it.
"""


def read_external_sheet(raw_sheet, sheet_name, file_name):
    """Returns (rows, note). `rows` is a list of dicts ready to become a
    DataFrame, empty if this sheet was refused entirely; `note` explains
    what happened, for the run's report.

    Refusing the whole sheet is the only outcome for a shape this does not
    recognize - there is no header-shape guess left to make, so there is
    nothing partial to salvage the way an inferred shape sometimes had.
    """
    header_rows = find_index_header_rows(raw_sheet)
    if len(header_rows) != 2:
        return [], (f'expected 2 "index" header rows (one per block), '
                    f"found {len(header_rows)} - skipped")

    blocks = split_blocks(raw_sheet, header_rows)
    data_rows, data_note = read_data_block(*blocks[0])
    if not data_rows:
        return [], data_note

    source_table = read_source_block(*blocks[1])
    data_table = pd.DataFrame(data_rows)
    data_table["Year"] = data_table["Year"].astype(float)
    merged = data_table.merge(source_table, on=["Indicator", "Year"], how="left")
    merged["Year"] = merged["Year"].astype(int)

    rows = merged.to_dict("records")
    with_source = sum(1 for r in rows if pd.notna(r["Source"]))
    note = f"{data_note} - {with_source:,} row(s) matched to a source citation"
    return rows, note


## Reshaping into the questionnaire layout

`build_questionnaire_records()` and `write_questionnaire_sheet()` below
follow the `reshape-external-data-questionnaire-layout` skill: they group
the rows `extract_sheet()` already produced into
`Index | Indicator | Country | Breakdown | year-columns` records - `Index`
assigned once per distinct indicator, in the order it was first seen, the
same convention the reference questionnaire files use - and write one styled
sheet per source table. `save_questionnaire_layout()` writes one such
workbook per source file, beside the source file.

In [ ]:
"""
CELL: Reshaping one file's already-extracted rows into the questionnaire
layout, and saving that as its own workbook beside the source file.
"""

_HEADER_FONT = Font(name="Calibri", size=11, bold=True)
_TITLE_FONT = Font(name="Calibri", size=12, bold=True)
_BODY_FONT = Font(name="Calibri", size=11)
_CENTER = Alignment(horizontal="center", vertical="center", wrap_text=True)
_LEFT = Alignment(horizontal="left", vertical="center")
_THIN = Side(style="thin", color="B7B7B7")
_BORDER = Border(left=_THIN, right=_THIN, top=_THIN, bottom=_THIN)
_HEADER_FILL = PatternFill("solid", fgColor="D9E1F2")


def build_questionnaire_records(rows):
    """Groups one sheet's already-extracted rows into the skill's record
    shape - one record per (Indicator, Sex, Country), a year->value map.
    Built from the SAME rows appended to the long file, never a second
    reading of the sheet, so this can never disagree with what was appended.
    """
    index_by_indicator = {}
    grouped = OrderedDict()
    for row in rows:
        indicator = row["Indicator"]
        if indicator not in index_by_indicator:
            index_by_indicator[indicator] = len(index_by_indicator) + 1
        key = (indicator, row["Sex"], row["Country"])
        grouped.setdefault(key, {})[row["Year"]] = row["Value"]

    records = []
    for (indicator, sex, country), values in grouped.items():
        records.append({
            "index": index_by_indicator[indicator], "indicator": indicator,
            "country": country, "breakdown": sex, "values": values,
        })
    years = sorted({row["Year"] for row in rows})
    return records, years


def write_questionnaire_sheet(out_wb, sheet_name, title, records, breakdown_label, years):
    """One sheet in the questionnaire layout: title row, blank row, bold
    bordered header row, one data row per record, frozen panes below the
    header. Styling only - see build_questionnaire_records() for the data.
    """
    ws = out_wb.create_sheet(sheet_name[:31])

    has_breakdown = breakdown_label is not None and any(r.get("breakdown") for r in records)
    fixed_cols = ["Index", "Indicator", "Country"] + ([breakdown_label] if has_breakdown else [])
    n_fixed = len(fixed_cols)
    total_cols = n_fixed + len(years)

    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=max(total_cols, 1))
    tcell = ws.cell(row=1, column=1, value=title)
    tcell.font = _TITLE_FONT
    tcell.alignment = _LEFT

    header_row = 3
    for i, colname in enumerate(fixed_cols, start=1):
        c = ws.cell(row=header_row, column=i, value=colname)
        c.font, c.alignment, c.fill, c.border = _HEADER_FONT, _CENTER, _HEADER_FILL, _BORDER
    for j, yr in enumerate(years, start=n_fixed + 1):
        c = ws.cell(row=header_row, column=j, value=yr)
        c.font, c.alignment, c.fill, c.border = _HEADER_FONT, _CENTER, _HEADER_FILL, _BORDER

    r = header_row + 1
    for rec in sorted(records, key=lambda rc: (rc["index"], rc["country"], rc.get("breakdown") or "")):
        ic = ws.cell(row=r, column=1, value=rec["index"])
        ic.font, ic.alignment, ic.border = _BODY_FONT, _CENTER, _BORDER
        c2 = ws.cell(row=r, column=2, value=rec["indicator"])
        c2.font, c2.alignment, c2.border = _BODY_FONT, _LEFT, _BORDER
        c3 = ws.cell(row=r, column=3, value=rec["country"])
        c3.font, c3.alignment, c3.border = _BODY_FONT, _LEFT, _BORDER
        col_offset = n_fixed
        if has_breakdown:
            cb = ws.cell(row=r, column=4, value=rec.get("breakdown"))
            cb.font, cb.alignment, cb.border = _BODY_FONT, _LEFT, _BORDER
        for j, yr in enumerate(years, start=col_offset + 1):
            cell = ws.cell(row=r, column=j, value=rec["values"].get(yr))
            cell.font = _BODY_FONT
            cell.alignment = Alignment(horizontal="center")
            cell.border = _BORDER
        r += 1

    ws.column_dimensions["A"].width = 8
    ws.column_dimensions["B"].width = 55
    ws.column_dimensions["C"].width = 20
    start_idx = 4
    if has_breakdown:
        ws.column_dimensions["D"].width = 16
        start_idx = 5
    for j in range(start_idx, total_cols + 1):
        ws.column_dimensions[get_column_letter(j)].width = 10
    ws.freeze_panes = ws.cell(row=header_row + 1, column=n_fixed + 1).coordinate
    return ws


def save_questionnaire_layout(chapter, source_path, sheet_rows):
    """One workbook, one sheet per source sheet, saved beside the source
    file. `sheet_rows` is {sheet_name: (rows, title)} for every sheet this
    notebook actually extracted something from - a sheet that was refused
    entirely has no rows and is not in it."""
    if not sheet_rows:
        return None
    out_wb = openpyxl.Workbook()
    out_wb.remove(out_wb.active)
    for sheet_name, (rows, title) in sheet_rows.items():
        records, years = build_questionnaire_records(rows)
        has_sex = any(r["Sex"] for r in rows)
        write_questionnaire_sheet(out_wb, sheet_name, title or sheet_name, records,
                                  "Sex" if has_sex else None, years)
    out_path = source_path.with_name(f"{source_path.stem}_reshaped.xlsx")
    out_wb.save(out_path)
    logger.info(f"  {chapter}: questionnaire-layout reshape -> {out_path.name} "
               f"({len(sheet_rows)} sheet(s))")
    return out_path


## Reading a chapter's external-data folder

In [ ]:
"""
CELL: read_external_file() - every sheet in one workbook, extracted or
refused, plus the reshaped questionnaire-layout copy saved beside it.
"""

SKIPPED_SHEETS = []     # {chapter, file, sheet, detail} - every sheet this refused entirely


def read_external_file(path, chapter):
    """Every usable row from every sheet in one external-data workbook."""
    try:
        xls = pd.ExcelFile(path, engine="openpyxl")
    except Exception as error:
        SKIPPED_SHEETS.append({
            "chapter": chapter, "file": path.name, "sheet": "-",
            "detail": f"could not open the file: {type(error).__name__}: {error}",
        })
        return []

    all_rows = []
    sheet_rows = {}
    for sheet_name in xls.sheet_names:
        try:
            raw_sheet = pd.read_excel(xls, sheet_name=sheet_name, header=None)
        except Exception as error:
            SKIPPED_SHEETS.append({
                "chapter": chapter, "file": path.name, "sheet": sheet_name,
                "detail": f"could not read the sheet: {type(error).__name__}: {error}",
            })
            continue

        rows, note = read_external_sheet(raw_sheet, sheet_name, path.name)
        logger.info(f"  {path.name} | {sheet_name}: {note}")
        if not rows:
            SKIPPED_SHEETS.append({
                "chapter": chapter, "file": path.name, "sheet": sheet_name, "detail": note,
            })
            continue
        all_rows.extend(rows)
        sheet_rows[sheet_name] = (rows, collapse_whitespace(rows[0]["Indicator"]))

    save_questionnaire_layout(chapter, path, sheet_rows)
    return all_rows


def read_external_chapter(chapter):
    """Every usable row from every .xlsx in one chapter's external-data folder."""
    folder = EXTERNAL_DATA_PATH / chapter
    files = sorted(f for f in folder.glob("*.xlsx")
                   if not f.name.startswith("~$") and not f.stem.endswith("_reshaped"))
    logger.info(f"{chapter}: {len(files)} external file(s)")

    rows = []
    for path in files:
        rows.extend(read_external_file(path, chapter))
    if not rows:
        return pd.DataFrame(columns=["Country", "Year", "Value", "Indicator", "Sex", "Source"])

    table = pd.DataFrame(rows)
    table["Chapter"] = chapter
    table[ORIGIN_COLUMN_EN] = ORIGIN_EXTERNAL_EN
    return table


## Run - part 1: read, append to English, find the gaps

In [ ]:
"""
CELL: Main run - reshape every chapter's external files, write each one's
questionnaire-layout copy, and save the extracted rows to
<Chapter>_EN_external.xlsx for notebook 4 to translate and append.
"""
SKIPPED_SHEETS.clear()

chapters = external_chapters()
print(f"Chapters with external data: {chapters}\n")

WRITTEN = {}   # chapter -> path written, for the summary below

for chapter in chapters:
    print(f"=== {chapter} ===")
    new_rows = read_external_chapter(chapter)
    if new_rows.empty:
        print(f"  nothing usable found\n")
        continue

    out_path = external_file_path(chapter)
    new_rows.to_excel(out_path, index=False, engine="openpyxl")
    WRITTEN[chapter] = out_path
    logger.info(f"  {chapter}: {len(new_rows):,} row(s) -> {out_path.name}")
    print()

print("=" * 70)
print(f"{sum(1 for _ in WRITTEN):,} chapter(s) written: "
     f"{', '.join(f'{c} ({p.name})' for c, p in WRITTEN.items()) or 'none'}")
print(f"{len(SKIPPED_SHEETS)} sheet(s) skipped entirely - see below")

if SKIPPED_SHEETS:
    print("\nSkipped entirely:")
    for row in SKIPPED_SHEETS:
        print(f"  {row['chapter']} \u00b7 {row['file']} \u00b7 {row['sheet']}: {row['detail']}")

records = [{"kind": "sheet skipped", **row} for row in SKIPPED_SHEETS]
paths, count = save_inconsistencies("3. EXTERNAL DATA", records, chapters=chapters)
print(f"\n{count} inconsistency(ies) recorded across {len(paths)} file(s)")
